In [1]:
import micropip
await micropip.install("tabulate")
await micropip.install("pandas")

import math
import pandas as pd

In [2]:
MICROSECOND = 1
MILLISECOND = 1000 * MICROSECOND
SECOND      = 1000 * MILLISECOND
MINUTE      = 60 * SECOND
HOUR        = 60 * MINUTE
DAY         = 24 * HOUR
MONTH       = 30 * DAY # Approximation as months are irregular
YEAR        = 365 * DAY # Defining YEAR in terms of days is a better approximation. 12 months would give 360 days. Note however that some years (leap years) have 366 days.
CENTURY     = 100 * YEAR # Somewhat inacurate as it ignores leap years

TIMES = {
	"1 second": SECOND,
	"1 minute": MINUTE,
	"1 hour": HOUR,
	"1 day": DAY,
	"1 month": MONTH,
	"1 year": YEAR,
	"1 century": CENTURY,
}

In [39]:
def inverse_n_log_n(t: int) -> int:
	"""
	 Inverts the function n*lg(n)	

	 From https://math.stackexchange.com/a/1302746
	 n*lg(n) = t => n = t/lg(n)
	 We start with a guess for n set to t, run the 
	 guess through the rearranged function and check if we get n back.
	 If not, set n to the returned value.
	 Repeat the process until we get the same n back
	"""
	n = float(t)
	while True:
		next = t/math.log2(n)
		if int(next) == int(n):
			return int(next)
		n = next

In [40]:
def inverse_factorial(t: int) -> int:
	"""
	Inverts the factorial function

	If the result would not be an integer, it gives the next integer.
	e.g. 4! = 24, and inverseFactorial(24) = 4
	however inverseFactorial(25) = 5 even though 5! = 120
	"""
	n = 1
	r = t
	while r > 1:
		n+=1
		r/=n
	return n

In [ ]:
def approximate_power_of_two(t):
	"""
	Aproximates the value of a large power of two in scientific notation

	The idea is that we want to represent the solution as some coeficient multiplied by 10 elevated to some exponent
	In other words, we want to find x, such that 2^t ~= 10^x
	By taking log base 10 on both sides we get: x = log10(2^t) = t * log10(2)
	"""
	# Calculate the exact base-10 exponent
	x = t * math.log10(2)
	
	# Split into integer and fractional parts
	exponent = int(x)
	fraction = x - exponent
	
	# Calculate the leading coefficient (multiplier)
	coefficient = 10 ** fraction
	
	return f"{coefficient:.4f}e+{exponent}"


In [105]:
def to_string(n):
	if isinstance(n, str):
		return n
	elif isinstance(n, int):
		return f"{n:e}" if n>10**5 else f"{n}"
	elif isinstance(n, float):
		return f"{int(n):e}" if n>10**5 else f"{int(n)}"
	else:
		raise ValueError("Invalid type.")

In [106]:
FUNCTIONS = {
	"lg(n)":   lambda n: math.log2(n),
	"sqrt(n)": lambda n: math.sqrt(n),
	"n":       lambda n: n,
	"n*lg(n)": lambda n: n * math.log2(n),
	"n^2":     lambda n: n**2,
	"n^3":     lambda n: n**3,
	"2^n":     lambda n: 2**n,
	"n!":      lambda n: math.factorial(n)
}


INVERTS = {
	# "lg(n)":   lambda t: 2**t, # I took this out because the numbers get so large that pyodide was strugling to compute them
	"lg(n)":   lambda t: approximate_power_of_two(t),
	"sqrt(n)": lambda t: t**2,
	"n":       lambda t: t,
	"n*lg(n)": lambda t: inverse_n_log_n(t),
	"n^2":     lambda t: math.sqrt(t),
	"n^3":     lambda t: math.cbrt(t),
	"2^n":     lambda t: math.log2(t),
	"n!":      lambda t: inverse_factorial(t),
}

In [107]:
grid = {}
for f_name, func in INVERTS.items():
	row = {}
	for t_name, time in TIMES.items():
		row[t_name] = to_string(func(time))
	grid[f_name] = row

df = pd.DataFrame.from_dict(grid, orient="index")
print(df.to_markdown())

|         | 1 second       | 1 minute         | 1 hour             | 1 day               | 1 month              | 1 year                | 1 century               |
|:--------|:---------------|:-----------------|:-------------------|:--------------------|:---------------------|:----------------------|:------------------------|
| lg(n)   | 9.9007e+301029 | 5.4934e+18061799 | 2.4566e+1083707984 | 2.3333e+26008991625 | 1.0947e+780269748761 | 2.0443e+9493281943259 | 1.3335e+949328194325931 |
| sqrt(n) | 1.000000e+12   | 3.600000e+15     | 1.296000e+19       | 7.464960e+21        | 6.718464e+24         | 9.945193e+26          | 9.945193e+30            |
| n       | 1.000000e+06   | 6.000000e+07     | 3.600000e+09       | 8.640000e+10        | 2.592000e+12         | 3.153600e+13          | 3.153600e+15            |
| n*lg(n) | 62746          | 2.801417e+06     | 1.333781e+08       | 2.755148e+09        | 7.187086e+10         | 7.976339e+11          | 6.861096e+13            |
| n^2     | 1000